# 🛡️ Kaggle 24/7 OSINT Agent + Tunnel Cloudflare & Keep-Alive
Plateforme OSINT 24/7 alimentée par Qwen3.6-12B GGUF. Boucle infinie active pour empêcher l'arrêt du container.

In [ ]:
# 1. Redirection TMPDIR & Initialisation des dossiers
import os

os.environ['TMPDIR'] = '/kaggle/working/tmp'
os.environ['PIP_CACHE_DIR'] = '/kaggle/working/tmp/pip'
os.makedirs('/kaggle/working/tmp', exist_ok=True)
os.makedirs('/kaggle/working/models', exist_ok=True)
print('🟢 Dossiers temporaires et modèles prêts !')

In [ ]:
# 2. Clonage ou Mise à jour par Git Pull
import os, subprocess

repo_dir = '/kaggle/working/projet_osint'
clone_url = 'https://github.com/whbky6vqjb-coder/osint.git'

if os.path.exists(repo_dir):
    print('⚡ Dépôt déjà présent : Exécution d\'un Git Pull (1 sec)...')
    subprocess.run(['git', '-C', repo_dir, 'pull', 'origin', 'main'])
    print('🟢 Code source mis à jour !')
else:
    print('📥 Premier clonage du dépôt Git...')
    subprocess.run(['git', 'clone', clone_url, repo_dir])
    print('🟢 Dépôt Git cloné !')

!pip install --no-cache-dir --prefer-binary huggingface_hub "llama-cpp-python[server]"

In [ ]:
import os
import stat
import urllib.request
import subprocess
import time
from huggingface_hub import hf_hub_download

# 1. Download Qwen GGUF model into models directory
model_dir = '/kaggle/working/models'
os.makedirs(model_dir, exist_ok=True)
model_filename = 'qwen2.5-coder-7b-instruct-q4_k_m.gguf'
model_path = os.path.join(model_dir, model_filename)

if not os.path.exists(model_path):
    print('📥 Downloading Qwen GGUF model from Hugging Face...')
    try:
        model_path = hf_hub_download(
            repo_id='Qwen/Qwen2.5-Coder-7B-Instruct-GGUF',
            filename='qwen2.5-coder-7b-instruct-q4_k_m.gguf',
            local_dir=model_dir
        )
        print(f'🟢 Model downloaded successfully at: {model_path}')
    except Exception as e:
        print(f'⚠️ HuggingFace download error: {e}, using fallback search...')

# 2. Launch llama_cpp.server on port 8080 with GPU offloading
print('🚀 Starting llama_cpp.server on 127.0.0.1:8080 with GPU layers...')
with open('/tmp/llama_server.log', 'w') as llama_log:
    subprocess.Popen([
        'python3', '-m', 'llama_cpp.server',
        '--model', model_path,
        '--host', '127.0.0.1',
        '--port', '8080',
        '--n_gpu_layers', '99',
        '--n_ctx', '8192'
    ], stdout=llama_log, stderr=subprocess.STDOUT)

time.sleep(10)

# 3. Ensure cloudflared binary is installed
cloudflared_bin = '/tmp/cloudflared'
if not os.path.exists(cloudflared_bin):
    print('Downloading cloudflared binary...')
    url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    urllib.request.urlretrieve(url, cloudflared_bin)
    os.chmod(cloudflared_bin, 0o755)
    print('cloudflared binary downloaded and set as executable.')

# 4. Launch cloudflared tunnel
print('🌐 Launching Cloudflare Tunnel on port 8080...')
with open('/tmp/cloudflared.log', 'w') as log_file:
    subprocess.Popen([cloudflared_bin, 'tunnel', '--url', 'http://127.0.0.1:8080'], stdout=log_file, stderr=subprocess.STDOUT)

time.sleep(10)

# 5. Run publisher script to push Cloudflare URL to GitHub
repo_dir = '/kaggle/working/projet_osint'
script_path = os.path.join(repo_dir, 'backend/app/cloud_sync/publish_llm_url.py')

if os.path.exists(script_path):
    print(f'Running publisher script at {script_path}...')
    subprocess.run(['python3', script_path], cwd=repo_dir)
else:
    print(f'Warning: Publisher script not found at {script_path}')


In [ ]:
# 5. Boucle d'exécution continue 24/7 avec Watchdog 11 heures
import time
import subprocess
import os

print("🟢 Serveur actif 24/7. Boucle d'écoute et Watchdog 11h activés !")
start_time = time.time()
eleven_hours = 11 * 3600  # 11 heures en secondes

counter = 0
while True:
    time.sleep(60)
    elapsed = time.time() - start_time
    counter += 1
    
    if elapsed >= eleven_hours:
        print("⏰ Limite de 11h atteinte ! Envoi du signal de redémarrage programmé à Hermes...")
        repo_dir = '/kaggle/working/projet_osint'
        storage_dir = os.path.join(repo_dir, 'storage')
        os.makedirs(storage_dir, exist_ok=True)
        
        maintenance_msg = "MAINTENANCE: Kaggle GPU server auto-restarting after 11 hours uptime.
"
        with open(os.path.join(storage_dir, 'llm_url.txt'), 'w', encoding='utf-8') as f:
            f.write(maintenance_msg)
        
        subprocess.run(['git', '-C', repo_dir, 'config', 'user.name', 'Kaggle-Bot'], check=False)
        subprocess.run(['git', '-C', repo_dir, 'config', 'user.email', 'kaggle-bot@osint.internal'], check=False)
        subprocess.run(['git', '-C', repo_dir, 'add', 'storage/llm_url.txt'], check=False)
        subprocess.run(['git', '-C', repo_dir, 'commit', '-m', 'chore: 11h scheduled Kaggle GPU maintenance restart'], check=False)
        subprocess.run(['git', '-C', repo_dir, 'push', 'origin', 'main'], check=False)
        
        print("✅ Signal de maintenance 11h publié sur GitHub avec succès !")
        break

    if counter % 30 == 0:
        remaining_hours = round((eleven_hours - elapsed) / 3600, 1)
        print(f'[{time.strftime("%Y-%m-%d %H:%M:%S")} 🟢] Le GPU tourne depuis {counter} min. ({remaining_hours}h restantes avant redémarrage 11h).')
